# Longformer

In [1]:
import torch
import torch.nn as nn
import numpy as np
import math

In [5]:
class long(nn.Module):
    def __init__(self, hidden_dim, slider):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.slider = slider

        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):

        batch_size, seq_len, hidden_dim = x.shape
        
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        outputs = []
        
        for i in range(seq_len):
            start = max(0, i - self.slider)
            end = i + 1 
            q = Q[:, i:i + 1, :] # for sequence dimension preservation. Q[:, i+1, :] means shape will become [4,128] but we want [4, 1, 128]
            k = K[:, start:end, :]
            v = V[:, start:end, :]
            
            scores = q @ k.transpose(-2,-1)
            scores = scores / math.sqrt(k.shape[-1])
            wts = torch.softmax(scores, dim = -1)
            output = wts @ v
            outputs.append(output)
        return torch.cat(outputs, dim = 1)

In [6]:
model = long(128, 3)
x = torch.rand(4,20,128)
model(x)

tensor([[[ 1.1635e-01,  1.5148e-01, -3.6484e-01,  ..., -7.9116e-02,
           2.8303e-01, -2.6156e-01],
         [ 1.7613e-01,  1.6585e-01, -2.3025e-01,  ..., -1.5675e-01,
           2.1097e-01, -1.9941e-01],
         [ 1.6447e-01,  2.1483e-01, -1.8354e-01,  ..., -1.4241e-01,
           2.0066e-01, -1.8783e-01],
         ...,
         [ 1.8164e-01,  1.5213e-01, -2.0382e-01,  ..., -1.3271e-02,
           2.7761e-01, -2.6331e-01],
         [ 1.3238e-01,  2.2141e-01, -1.3545e-01,  ...,  2.9964e-02,
           2.5218e-01, -2.6101e-01],
         [ 7.4298e-02,  2.3793e-01, -1.8312e-01,  ...,  6.7695e-02,
           2.1774e-01, -1.9392e-01]],

        [[ 1.9601e-02,  1.9654e-01, -1.4435e-01,  ..., -2.2360e-01,
          -3.6878e-02, -3.1374e-01],
         [ 5.5902e-02,  1.7156e-01, -1.6646e-01,  ...,  7.9085e-02,
           6.5259e-02, -3.7508e-01],
         [ 6.4733e-02,  1.7990e-01, -2.1904e-01,  ...,  8.6702e-02,
          -2.2781e-03, -3.4923e-01],
         ...,
         [-1.4710e-01,  2

# Performer

In [9]:
class perf(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)

    def phi(self, x):
        return torch.nn.functional.elu(x) + 1
    
    def forward(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q_dash = self.phi(Q)
        K_dash = self.phi(K)

        KV = K_dash.transpose(-2,-1) @ V
        output = Q_dash @ KV
        return output

In [10]:
model = perf(128)
x = torch.rand(4,20,128)
model(x)

tensor([[[ -354.7364, -1162.4957,    60.7086,  ...,  -847.3179,
           -193.6107,  -827.8726],
         [ -358.5833, -1175.4923,    61.7738,  ...,  -855.9498,
           -195.1985,  -837.3957],
         [ -360.6477, -1180.8724,    61.7680,  ...,  -860.5517,
           -196.4828,  -841.8120],
         ...,
         [ -353.1567, -1157.9839,    60.2993,  ...,  -843.6860,
           -192.7373,  -824.4002],
         [ -359.2520, -1176.6481,    61.2945,  ...,  -857.5068,
           -196.3142,  -838.3502],
         [ -349.9593, -1146.5719,    60.0687,  ...,  -835.4357,
           -190.7681,  -816.9335]],

        [[ -310.7058,  -938.1143,  -137.9260,  ...,  -843.5891,
           -196.9639,  -642.3606],
         [ -317.7110,  -959.0750,  -140.9323,  ...,  -862.2825,
           -201.1549,  -656.7418],
         [ -313.5576,  -947.7095,  -139.2499,  ...,  -852.2559,
           -198.9537,  -648.8535],
         ...,
         [ -310.8501,  -938.0529,  -137.6138,  ...,  -843.5018,
           -196

# Linformer

In [11]:
class lin(nn.Module):
    def __init__(self, hidden_dim, seq_len, low_seq_dim):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.seq_len = seq_len
        self.low_seq_dim = low_seq_dim
        
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)
        self.Wv = nn.Linear(hidden_dim, hidden_dim)
        self.E = nn.Parameter(torch.rand(seq_len, low_seq_dim))
        self.F = nn.Parameter(torch.rand(seq_len, low_seq_dim))

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        K = torch.matmul(self.E[:seq_len].T, K)
        V = torch.matmul(self.F[:seq_len].T, V)

        scores = Q @ K.transpose(-2,-1)
        scores = scores / math.sqrt(K.shape[-1])
        wts = torch.softmax(scores, dim = -1)
        output = wts @ V
        return output

In [12]:
model = lin(128, 20, 10)
x = torch.rand(4,20,128)
model(x)

tensor([[[ 2.0931,  5.4132, -3.6469,  ..., -2.3634,  4.3500, -1.0323],
         [ 2.0821,  5.3892, -3.6284,  ..., -2.3546,  4.3298, -1.0275],
         [ 2.0824,  5.3922, -3.6318,  ..., -2.3540,  4.3335, -1.0298],
         ...,
         [ 2.0858,  5.3921, -3.6330,  ..., -2.3561,  4.3354, -1.0322],
         [ 2.0804,  5.3817, -3.6293,  ..., -2.3541,  4.3280, -1.0347],
         [ 2.0839,  5.3889, -3.6281,  ..., -2.3578,  4.3290, -1.0279]],

        [[ 1.7190,  4.0335, -3.1325,  ..., -2.4002,  3.9124, -0.8645],
         [ 1.7066,  4.0258, -3.1301,  ..., -2.4021,  3.9068, -0.8581],
         [ 1.7057,  4.0182, -3.1217,  ..., -2.3989,  3.8971, -0.8594],
         ...,
         [ 1.7091,  4.0451, -3.1377,  ..., -2.4035,  3.9244, -0.8656],
         [ 1.7012,  4.0157, -3.1205,  ..., -2.3952,  3.8961, -0.8584],
         [ 1.6950,  4.0347, -3.1347,  ..., -2.4077,  3.9153, -0.8563]],

        [[ 1.8667,  5.3528, -2.7030,  ..., -2.5631,  4.7214, -0.4960],
         [ 1.8680,  5.3594, -2.6948,  ..., -2